# 04x — Full-run convergence diagnosis (read-only, no sampling)

Reads the saved FULL Model D and Model E posteriors from the debugged Notebook 04 run and locates **which galaxies and parameters fail to mix**.
It also checks whether the chains disagree (multiple modes) or are merely slow. Nothing is refitted and no existing file is modified.
New CSVs are written next to the run folder, under `diagnosis_04x/`. Runtime: a few minutes on Colab or Kaggle CPU.

**Set `RUN_DIR`** to the unzipped run folder, e.g. `/content/drive/MyDrive/RAR_Universality_Research/05_Results/Diagnostics/04_GP_Debugged/20261026T083032Z`.
On Colab, mount Drive first (the first cell does it).

In [ ]:
import os, sys, glob, json
from pathlib import Path
if "google.colab" in sys.modules:
    from google.colab import drive; drive.mount("/content/drive")
RUN_DIR = Path(os.environ.get("RAR_RUN_DIR", "/content/drive/MyDrive/RAR_Universality_Research/05_Results/Diagnostics/04_GP_Debugged/20261026T083032Z"))
if not RUN_DIR.is_dir():   # Kaggle or other layouts: search for the newest run folder that has full posteriors
    hits = sorted(glob.glob("/**/04_GP_Debugged/*/posterior/04d_gp_full_model_D_posterior*.nc", recursive=True))
    if not hits: raise FileNotFoundError(f"{RUN_DIR} not found; set RUN_DIR to the unzipped run folder")
    RUN_DIR = Path(hits[-1]).parents[1]
print("Run folder:", RUN_DIR)
import subprocess, importlib.util
for m in ("xarray", "h5netcdf", "h5py"):
    if importlib.util.find_spec(m) is None: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", m])
import numpy as np, pandas as pd, xarray as xr, scipy.stats as sst
OUTD = RUN_DIR / "diagnosis_04x"; OUTD.mkdir(exist_ok=True)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

## Diagnostics (Vehtari et al. 2021)

Rank-normalised split R-hat, including the folded version, plus bulk and tail ESS. These are the same definitions as in Notebook 04.

In [ ]:
def _ess_raw(x):                       # x: (chains, draws); Geyer initial-monotone-sequence ESS
    m, n = x.shape
    if np.var(x) == 0: return float(m * n)
    xc = x - x.mean(1, keepdims=True)
    f = np.fft.rfft(xc, n=2 * n); acov = np.fft.irfft(f * np.conj(f))[:, :n] / n
    var_w = acov[:, 0].mean() * n / (n - 1); var_p = var_w * (n - 1) / n + (np.var(x.mean(1), ddof=1) if m > 1 else 0)
    rho = 1 - (var_w - acov.mean(0)) / var_p; rho[0] = 1
    t, s, prev = 1, 0.0, np.inf
    while t + 1 < n:
        p = rho[t] + rho[t + 1]
        if p < 0: break
        p = min(p, prev); s += p; prev = p; t += 2
    tau = -1 + 2 * (s + rho[0]) if s > 0 else 1.0
    return float(m * n / max(tau, 1e-12))
def _split(x): n = x.shape[1] // 2; return np.concatenate([x[:, :n], x[:, -n:]], 0)
def _z(x): r = sst.rankdata(x.ravel()).reshape(x.shape); return sst.norm.ppf((r - 3 / 8) / (x.size + 1 / 4))
def _rhat(x):
    m, n = x.shape; w = x.var(1, ddof=1).mean(); b = n * x.mean(1).var(ddof=1)
    return float(np.sqrt(((n - 1) / n * w + b / n) / w)) if w > 0 else np.nan
def rhat(x): xs = _split(x); return max(_rhat(_z(xs)), _rhat(_z(np.abs(xs - np.median(xs)))))
def ess_bulk(x): return _ess_raw(_z(_split(x)))
def ess_tail(x):
    q5, q95 = np.quantile(x, [.05, .95]); xs = _split(x)
    return min(_ess_raw((xs <= q5).astype(float)), _ess_raw((xs >= q95).astype(float)))

In [ ]:
GLOBALS = ["log10_a0", "beta", "sigma_int", "tau", "mu_lnA", "sigma_lnA", "mu_u", "sigma_u"]
PER_GAL = ["zA", "zu", "zD", "zI", "zUd", "A", "ell", "D", "inc"]
SUMMARY = {}
for model in ("D", "E"):
    f = sorted((RUN_DIR / "posterior").glob(f"04d_gp_full_model_{model}_posterior*.nc"))
    if not f: print(f"Model {model}: no full posterior found"); continue
    post = xr.open_dataset(f[-1], group="posterior", engine="h5netcdf"); st = xr.open_dataset(f[-1], group="sample_stats", engine="h5netcdf")
    C, S = post.sizes["chain"], post.sizes["draw"]
    div = st["diverging"].values.astype(bool); steps = st["n_steps"].values; en = st["energy"].values
    run = {"model": model, "chains": C, "draws_per_chain": S, "divergences": int(div.sum()), "divergences_per_chain": div.sum(1).tolist(),
           "tree_depth_hits": int((steps >= 1023).sum()), "mean_steps": float(steps.mean()),
           "accept_per_chain": np.round(st["acceptance_rate"].values.mean(1), 3).tolist(),
           "step_size_per_chain": np.round(st["step_size"].values[:, -1], 4).tolist(),
           "bfmi_per_chain": np.round([np.mean(np.diff(e) ** 2) / np.var(e) for e in en], 3).tolist()}
    print(f"\n===== FULL Model {model} =====\n", json.dumps(run, indent=1))
    # global parameters, with per-chain medians to show whether chains agree
    g = []
    for k in GLOBALS:
        x = post[k].values
        g.append({"parameter": k, "r_hat": rhat(x), "ess_bulk": ess_bulk(x), "ess_tail": ess_tail(x),
                  "median": float(np.median(x)), **{f"chain{c}_median": float(np.median(x[c])) for c in range(C)}})
    gdf = pd.DataFrame(g); gdf["meets_criteria"] = (gdf.r_hat < 1.01) & (gdf.ess_bulk >= 400) & (gdf.ess_tail >= 400)
    print("Global parameters:"); display(gdf.round(4))
    gdf.to_csv(OUTD / f"04x_full_{model}_global_diagnostics.csv", index=False)
    # per-galaxy
    names = [str(n) for n in post.galaxy.values]; rows = []
    for gi, name in enumerate(names):
        r = {"galaxy": name}
        for k in PER_GAL:
            if k not in post: continue
            x = post[k].values[:, :, gi]
            r[f"rhat_{k}"] = rhat(x); r[f"essb_{k}"] = ess_bulk(x)
            if k in ("A", "ell", "D", "inc"):
                cm = np.median(x, axis=1); r[f"{k}_chain_medians"] = np.round(cm, 3).tolist()
                # between-chain spread of medians relative to the pooled posterior sd: >1 means chains sit in different places
                r[f"{k}_chain_spread_in_sd"] = float((cm.max() - cm.min()) / (x.std() + 1e-300))
        rk = [c for c in r if c.startswith("rhat_z")]; ek = [c for c in r if c.startswith("essb_z")]
        r["max_rhat"] = max(r[c] for c in rk); r["min_ess_bulk"] = min(r[c] for c in ek)
        rows.append(r)
    pg = pd.DataFrame(rows).sort_values("max_rhat", ascending=False)
    bad = pg[(pg.max_rhat >= 1.01) | (pg.min_ess_bulk < 400)]
    print(f"Galaxies failing R-hat < 1.01 or bulk ESS >= 400 on any sampled per-galaxy coordinate: {len(bad)} of {len(pg)}")
    cols = ["galaxy", "max_rhat", "min_ess_bulk", "rhat_zA", "rhat_zu", "rhat_zD", "rhat_zI", "A_chain_medians", "ell_chain_medians",
            "D_chain_medians", "inc_chain_medians", "A_chain_spread_in_sd", "ell_chain_spread_in_sd", "D_chain_spread_in_sd"]
    display(bad[[c for c in cols if c in bad]].head(25))
    pg.to_csv(OUTD / f"04x_full_{model}_per_galaxy_diagnostics.csv", index=False)
    multimodal = bad[(bad.get("ell_chain_spread_in_sd", 0) > 1) | (bad.get("A_chain_spread_in_sd", 0) > 1) | (bad.get("D_chain_spread_in_sd", 0) > 1)]
    SUMMARY[model] = {**run, "global_params_failing": gdf.loc[~gdf.meets_criteria, "parameter"].tolist(),
                      "n_galaxies_failing": int(len(bad)), "galaxies_failing": bad.galaxy.tolist(),
                      "n_failing_with_chains_in_different_places": int(len(multimodal)), "those_galaxies": multimodal.galaxy.tolist()}
json.dump(SUMMARY, open(OUTD / "04x_summary.json", "w"), indent=1, default=str)
print("\n===== SUMMARY =====\n", json.dumps(SUMMARY, indent=1, default=str)); print("Saved to", OUTD)